# UN Immigration Data: Data Cleaning and Transformation

**Purpose:** This notebook reads the United Nations immigration spreadsheet, cleans the annual values, and organizes the available records into a DataFrame.

The supplied workbook identifies **Canada** as the reporting country and contains **regional aggregate records**, not individual countries. Therefore, this file cannot produce the country-level example with `Country`, `Continent`, `Region`, and `DevName` columns. The notebook keeps the data faithful to the uploaded source rather than inventing missing country classifications.

## 1. Import the libraries

Pandas is used to read and organize the spreadsheet. NumPy is used to handle missing values.

In [ ]:
import pandas as pd
import numpy as np

## 2. Read the Excel file

The spreadsheet has title and source information above the actual table. The column headings are on Excel row 21, so `header=20` is used.

In [26]:

from google.colab import files
import pandas as pd
import numpy as np

# Select the Excel file from your computer
uploaded = files.upload()

# Automatically get the uploaded filename
file_name = list(uploaded.keys())[0]

# Read the Excel spreadsheet
raw_data = pd.read_excel(file_name, sheet_name="Sheet1", header=20)

print("File uploaded successfully!")
print("File name:", file_name)
print("Original data shape:", raw_data.shape)

display(raw_data.head())


Saving Sharmin_Akter_Assignmnet 3 Machine laearning UN Immegration data.xlsx to Sharmin_Akter_Assignmnet 3 Machine laearning UN Immegration data.xlsx
File uploaded successfully!
File name: Sharmin_Akter_Assignmnet 3 Machine laearning UN Immegration data.xlsx
Original data shape: (29, 38)


,Type,Coverage,AreaName,RegName,1980,1981,1982,1983,1984,1985,...,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
0,Immigrants,Citizens,Northern America,Northern America,..,..,..,..,..,..,...,..,..,..,..,..,..,..,1,1,NaN
1,Immigrants,Foreigners,Africa,Eastern Africa,1471,1641,1426,1094,1187,1134,...,7726,7083,6750,6669,6705,6922,8591,8956,8846,NaN
2,Immigrants,Foreigners,Africa,Middle Africa,33,27,44,32,42,30,...,2452,2802,2834,2951,3034,3255,3369,3146,4602,NaN
3,Immigrants,Foreigners,Africa,Northern Africa,1100,1268,1348,936,842,840,...,12478,11323,12705,11750,13456,16320,19271,15712,15453,NaN
4,Immigrants,Foreigners,Africa,Southern Africa,1041,1126,791,387,297,327,...,1203,1012,1148,1230,1184,1235,1306,1049,1348,NaN


In [27]:
from google.colab import files

uploaded = files.upload()

file_name = next(iter(uploaded))

raw_data = pd.read_excel(file_name, sheet_name="Sheet1", header=20)

print("File uploaded successfully!")
print("Original data shape:", raw_data.shape)

raw_data.head()

StopIteration: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clean the column names and select the relevant records

The year columns are converted to text labels. The analysis focuses on immigration records for foreigners. Rows labeled as regional totals are excluded so they are not counted together with their component regions.

In [35]:
# Standardize column labels
raw_data.columns = [str(column).replace(".0", "") for column in raw_data.columns]

# Select immigrant records for foreign nationals
immigration = raw_data[
    (raw_data["Type"] == "Immigrants") &
    (raw_data["Coverage"] == "Foreigners")
].copy()

# Remove subtotal rows such as "Africa Total"
immigration = immigration[
    ~immigration["AreaName"].astype(str).str.endswith(" Total")
].copy()

print("Records after filtering:", immigration.shape[0])
immigration[["Type", "Coverage", "AreaName", "RegName"]].head()

Records after filtering: 23


,Type,Coverage,AreaName,RegName
1,Immigrants,Foreigners,Africa,Eastern Africa
2,Immigrants,Foreigners,Africa,Middle Africa
3,Immigrants,Foreigners,Africa,Northern Africa
4,Immigrants,Foreigners,Africa,Southern Africa
5,Immigrants,Foreigners,Africa,Western Africa


## 4. Prepare the year columns

The symbol `..` in the source represents an unavailable value. It is replaced with a missing value, and annual columns are converted to numeric data.

In [29]:
year_columns = [str(year) for year in range(1980, 2014)]

for year in year_columns:
    immigration[year] = pd.to_numeric(
        immigration[year].replace("..", np.nan),
        errors="coerce"
    )

immigration[year_columns].head()

/tmp/ipykernel_500/3287859720.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  immigration[year].replace("..", np.nan),


,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,...,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
1,1471.0,1641.0,1426.0,1094.0,1187.0,1134.0,1454.0,2734.0,3237.0,4094.0,...,7726,7083,6750,6669,6705,6922,8591.0,8956.0,8846,NaN
2,33.0,27.0,44.0,32.0,42.0,30.0,61.0,100.0,91.0,119.0,...,2452,2802,2834,2951,3034,3255,3369.0,3146.0,4602,NaN
3,1100.0,1268.0,1348.0,936.0,842.0,840.0,1115.0,1864.0,2239.0,3574.0,...,12478,11323,12705,11750,13456,16320,19271.0,15712.0,15453,NaN
4,1041.0,1126.0,791.0,387.0,297.0,327.0,725.0,1480.0,1329.0,1319.0,...,1203,1012,1148,1230,1184,1235,1306.0,1049.0,1348,NaN
5,306.0,301.0,210.0,222.0,271.0,319.0,427.0,1316.0,656.0,788.0,...,4333,5303,5751,5684,5511,6802,8355.0,6578.0,7834,NaN


## 5. Create the cleaned DataFrame

`AreaName` is retained as the broad area and `RegName` as the region. The `Total` column is calculated by adding the available annual values from 1980 through 2013. Missing values are not treated as reported counts.

In [30]:
cleaned_data = immigration[["AreaName", "RegName"] + year_columns].copy()
cleaned_data = cleaned_data.rename(columns={
    "AreaName": "Continent",
    "RegName": "Region"
})

cleaned_data["Total"] = cleaned_data[year_columns].sum(axis=1, min_count=1)
cleaned_data = cleaned_data.set_index("Region")

cleaned_data.head(10)

,Continent,1980,1981,1982,1983,1984,1985,1986,1987,1988,...,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
Region,,,,,,,,,,,,,,,,,,,,,
Eastern Africa,Africa,1471.0,1641.0,1426.0,1094.0,1187.0,1134.0,1454.0,2734.0,3237.0,...,7083,6750,6669,6705,6922,8591.0,8956.0,8846,NaN,157941.0
Middle Africa,Africa,33.0,27.0,44.0,32.0,42.0,30.0,61.0,100.0,91.0,...,2802,2834,2951,3034,3255,3369.0,3146.0,4602,NaN,43623.0
Northern Africa,Africa,1100.0,1268.0,1348.0,936.0,842.0,840.0,1115.0,1864.0,2239.0,...,11323,12705,11750,13456,16320,19271.0,15712.0,15453,NaN,241469.0
Southern Africa,Africa,1041.0,1126.0,791.0,387.0,297.0,327.0,725.0,1480.0,1329.0,...,1012,1148,1230,1184,1235,1306.0,1049.0,1348,NaN,40135.0
Western Africa,Africa,306.0,301.0,210.0,222.0,271.0,319.0,427.0,1316.0,656.0,...,5303,5751,5684,5511,6802,8355.0,6578.0,7834,NaN,97237.0
Central Asia,Asia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1134,903,936,805,962,905.0,769.0,1029,NaN,14333.0
Eastern Asia,Asia,6836.0,8895.0,5481.0,3254.0,2624.0,2979.0,3416.0,5403.0,5887.0,...,50306,41763,35591,39602,37497,37954.0,35159.0,40449,NaN,800662.0
South-Eastern Asia,Asia,8693.0,9191.0,9022.0,8845.0,12161.0,9787.0,7801.0,10368.0,13542.0,...,22809,25105,27256,30614,34166,44199.0,40494.0,37695,NaN,653970.0
Southern Asia,Asia,11315.0,11566.0,11591.0,10157.0,9607.0,7654.0,12468.0,19947.0,20225.0,...,69510,66742,56032,54154,53083,62284.0,53673.0,60568,NaN,1331156.0


## 6. Check the result

The checks below show the final dimensions, column names, and missing-value counts.

In [31]:
print("Final DataFrame shape:", cleaned_data.shape)
print("\nColumns:")
print(cleaned_data.columns.tolist())

print("\nMissing values by column:")
print(cleaned_data.isna().sum())

cleaned_data

Final DataFrame shape: (23, 36)

Columns:
['Continent', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', 'Total']

Missing values by column:
Continent     0
1980          1
1981          2
1982          2
1983          1
1984          2
1985          2
1986          2
1987          2
1988          1
1989          1
1990          2
1991          2
1992          1
1993          0
1994          1
1995          0
1996          0
1997          1
1998          0
1999          0
2000          0
2001          0
2002          0
2003          0
2004          0
2005          0
2006          0
2007          0
2008          0
2009          0
2010          1
2011          1
2012          0
2013         23
Total         0
dtype: int64


,Continent,1980,1981,1982,1983,1984,1985,1986,1987,1988,...,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
Region,,,,,,,,,,,,,,,,,,,,,
Eastern Africa,Africa,1471.0,1641.0,1426.0,1094.0,1187.0,1134.0,1454.0,2734.0,3237.0,...,7083,6750,6669,6705,6922,8591.0,8956.0,8846,NaN,157941.0
Middle Africa,Africa,33.0,27.0,44.0,32.0,42.0,30.0,61.0,100.0,91.0,...,2802,2834,2951,3034,3255,3369.0,3146.0,4602,NaN,43623.0
Northern Africa,Africa,1100.0,1268.0,1348.0,936.0,842.0,840.0,1115.0,1864.0,2239.0,...,11323,12705,11750,13456,16320,19271.0,15712.0,15453,NaN,241469.0
Southern Africa,Africa,1041.0,1126.0,791.0,387.0,297.0,327.0,725.0,1480.0,1329.0,...,1012,1148,1230,1184,1235,1306.0,1049.0,1348,NaN,40135.0
Western Africa,Africa,306.0,301.0,210.0,222.0,271.0,319.0,427.0,1316.0,656.0,...,5303,5751,5684,5511,6802,8355.0,6578.0,7834,NaN,97237.0
Central Asia,Asia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1134,903,936,805,962,905.0,769.0,1029,NaN,14333.0
Eastern Asia,Asia,6836.0,8895.0,5481.0,3254.0,2624.0,2979.0,3416.0,5403.0,5887.0,...,50306,41763,35591,39602,37497,37954.0,35159.0,40449,NaN,800662.0
South-Eastern Asia,Asia,8693.0,9191.0,9022.0,8845.0,12161.0,9787.0,7801.0,10368.0,13542.0,...,22809,25105,27256,30614,34166,44199.0,40494.0,37695,NaN,653970.0
Southern Asia,Asia,11315.0,11566.0,11591.0,10157.0,9607.0,7654.0,12468.0,19947.0,20225.0,...,69510,66742,56032,54154,53083,62284.0,53673.0,60568,NaN,1331156.0


## 7. Save the cleaned data

The cleaned DataFrame is exported as a CSV file so it can also be opened in Excel.

In [36]:
output_file = "UN_Immigration_Cleaned_Regional.csv"
cleaned_data.to_csv(output_file)
print(f"Cleaned data saved as: {output_file}")

Cleaned data saved as: UN_Immigration_Cleaned_Regional.csv


## Conclusion

The spreadsheet was imported, filtered, cleaned, and reshaped into a regional immigration DataFrame. The supplied workbook contains regional information for Canada only. It does not include individual country records or a `DevName` field, so the country-level output shown in the assignment example requires a different or more complete source file.